# Leçon 07 - Modèle de Conception de Planification

Ce carnet montre le **Modèle de Conception de Planification** pour les agents IA en utilisant le Framework Agent de Microsoft.  
Vous apprendrez comment décomposer des demandes de voyage complexes en sous-tâches structurées, les attribuer à des agents spécialistes,  
et exécuter le plan résultant — le tout en utilisant une sortie structurée propulsée par des modèles Pydantic.


## Installation


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Décomposition de la Tâche

La décomposition de la tâche est le cœur du modèle de conception de planification. Au lieu de demander à un seul agent de gérer une requête complexe de bout en bout, nous divisons le problème en **sous-tâches** plus petites et bien définies. Chaque sous-tâche est assignée à un agent spécialiste (par exemple, vols, hôtels, activités) avec des priorités claires et un ordre de dépendance.

Cette approche offre plusieurs avantages :
- **Clarté** : chaque sous-tâche a une responsabilité unique.
- **Parallélisme** : les sous-tâches indépendantes peuvent s'exécuter simultanément.
- **Fiabilité** : les échecs sont isolés à des sous-tâches individuelles.
- **Suivi du budget** : les coûts sont estimés par sous-tâche et consolidés.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Création d'un agent de planification avec sortie structurée

L'agent de planification agit comme un **coordinateur à la réception**. Étant donné une demande de voyage de haut niveau, il produit un `TravelPlan` structuré — décomposant la demande en sous-tâches, fixant les priorités et identifiant les dépendances afin qu'un concierge ou une couche d'exécution puisse effectuer le travail.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Exécution d'un plan avec des outils spécialisés

Une fois que l'agent d'accueil a établi un plan structuré, l'**agent concierge** l'exécute.  
Chaque outil spécialisé gère une catégorie de sous-tâche (vols, hôtels, activités). Le concierge  
parcourt les sous-tâches du plan dans l'ordre des dépendances et envoie chacune d'elles à l'outil  
approprié.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Résumé

Dans cette leçon, vous avez appris le **Modèle de Conception de Planification** pour les agents IA :

1. **Décomposition de la tâche** — Un agent de planification à la réception décompose une demande de voyage complexe en sous-tâches structurées en utilisant des modèles Pydantic, en attribuant chacune à un agent spécialiste avec des priorités et des dépendances.
2. **Sortie structurée** — En passant un `response_format`, l'agent renvoie un objet `TravelPlan` validé au lieu d’un texte libre, ce qui rend le traitement en aval fiable.
3. **Exécution du plan** — Un agent concierge parcourt les sous-tâches en utilisant des outils spécialistes (`book_flight`, `reserve_hotel`, `book_activity`) pour réaliser le plan et rapporter les résultats.

Ce modèle sépare *ce qu’il faut faire* (la planification) de *la manière de le faire* (l’exécution), rendant les agents plus modulaires, testables et faciles à étendre.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertissement** :  
Ce document a été traduit à l’aide du service de traduction automatique [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous nous efforçons d’assurer l’exactitude, veuillez noter que les traductions automatiques peuvent contenir des erreurs ou des inexactitudes. Le document original dans sa langue d’origine doit être considéré comme la source faisant foi. Pour les informations critiques, une traduction professionnelle réalisée par un humain est recommandée. Nous déclinons toute responsabilité en cas de malentendus ou de mauvaises interprétations résultant de l’utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
